In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gemma4",
    model_provider="openai",
    api_key="dummy",
    base_url="http://localhost:8080/v1"
)

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [ ]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="asyncio, concurrency etc come under what"),
            AIMessage(
                content=(
                    "`asyncio` and concurrency fall under asynchronous programming and "
                    "concurrent computing (often grouped broadly under software architecture "
                    "and systems programming).\n\n"
                    "Primary umbrella categories include:\n"
                    "- Concurrent Computing / Concurrency: Managing multiple computations happening during overlapping periods.\n"
                    "- Asynchronous Programming (Async/Await): A cooperative execution model where tasks yield control while waiting for I/O rather than blocking.\n"
                    "- Non-Blocking I/O & Event-Driven Architecture: Operating on an event loop that wakes up coroutines when data is ready.\n"
                    "- Parallelism and Multitasking: Dividing workloads across CPU cores (multiprocessing/threads) versus single-threaded cooperative scheduling (event loops).\n\n"
                    "In practical terms, they sit under Operating Systems, Distributed Systems, Performance Optimization, Backend Engineering, and I/O Management."
                )
            ),
            HumanMessage(content="what perspective"),
            AIMessage(
                content=(
                    "Depending on the lens you view software engineering through, concurrency and asyncio represent different concepts:\n\n"
                    "- Operating Systems & Systems: Focuses on resource scheduling, kernel primitives (epoll, kqueue), thread context-switching overhead, and interrupts.\n"
                    "- Programming Language & Paradigm: Focuses on control flow, coroutines, generators, call stacks, and cooperative multitasking via async/await.\n"
                    "- Software Architecture: Focuses on system responsiveness, high-throughput connection handling, and event-driven patterns.\n"
                    "- Performance & Optimization: Focuses on resolving I/O-bound bottlenecks and eliminating idle CPU wait cycles.\n"
                    "- Correctness & Safety: Focuses on shared memory, atomicity, race conditions, deadlocks, and synchronization primitives.\n\n"
                    "Key contrast: Parallelism is a hardware perspective (simultaneous CPU execution), while concurrency is a software structure perspective (interleaved task management). Similarly, threads represent OS-level preemptive multitasking, whereas asyncio represents application-level cooperative multitasking."
                )
            ),
            HumanMessage(
                content="Replace this chat message history with the our message history. Dont add tables"
            ),
        ]
    },
    {"configurable": {"thread_id": "1"}},
)

pprint(response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nTo understand the conceptual categorization and different perspectives (e.g., OS, architectural, performance) under which asynchronous programming concepts like `asyncio` and concurrency fall.\n\n## SUMMARY\n\nConcurrency and `asyncio` are categorized under asynchronous programming and concurrent computing.\nKey conceptual distinctions noted:\n- **Concurrency** is a software structure perspective (interleaved task management).\n- **Parallelism** is a hardware perspective (simultaneous CPU execution).\n- **Threads** represent OS-level preemptive multitasking, whereas **`asyncio`** represents application-level cooperative multitasking.\nDifferent lenses offer varied perspectives:\n- **Operating Systems:** Focuses on scheduling, kernel primitives.\n- **Programming Language/Paradigm:** Focuses on coroutines, async/await, cooperative multitasking.\n- **Software Architecture:** Focuses 

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

To understand the conceptual categorization and different perspectives (e.g., OS, architectural, performance) under which asynchronous programming concepts like `asyncio` and concurrency fall.

## SUMMARY

Concurrency and `asyncio` are categorized under asynchronous programming and concurrent computing.
Key conceptual distinctions noted:
- **Concurrency** is a software structure perspective (interleaved task management).
- **Parallelism** is a hardware perspective (simultaneous CPU execution).
- **Threads** represent OS-level preemptive multitasking, whereas **`asyncio`** represents application-level cooperative multitasking.
Different lenses offer varied perspectives:
- **Operating Systems:** Focuses on scheduling, kernel primitives.
- **Programming Language/Paradigm:** Focuses on coroutines, async/await, cooperative multitasking.
- **Software Architecture:** Focuses on responsiveness and event-driven patterns.
- **Per

## Trim/delete messages

In [6]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [7]:
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [8]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='d55c8f31-9e86-4cf0-a991-0388e41f2033'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='a544743f-f226-458b-bdb4-f10a0e969d12', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='ed20b66d-f8ef-4d41-a19a-bd286a322387'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='f0059633-ff22-4182-a9c8-1f1f8382bec6', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='4b29cd5b-d85e-4788-8b2e-43a0f3bff923'),
              AIMessage(content="I do not have the ability to measure the temperature of your p

In [9]:
print(response["messages"][-1].content)

I do not have the ability to measure the temperature of your physical device.

Since you mentioned the device won't turn on, and you confirmed it is plugged in and appears to be powered, we need to narrow down the possibilities.

Could you tell me **what kind of device** it is (e.g., laptop, smartphone, tablet, monitor, etc.)?

In the meantime, here are some general troubleshooting steps, depending on what the device is:

### General Troubleshooting Steps:

1.  **Check the Power Source:**
    *   Try a different **wall outlet** to ensure the outlet itself is working.
    *   If you have another **power cable** that matches, try swapping it out to rule out a faulty cable.
    *   If there is a power brick, make sure the connection between the brick and the cable is secure.

2.  **Check the Power Button/Controls:**
    *   If it has a power button, try pressing and **holding** it down for an extended period (like 15-30 seconds). Sometimes this can force a cold boot or reset.

3.  **Check